In [ ]:
import numpy as np
G = -9.8

def distance_point_to_line_segment(p, a, b):
    """计算点p到线段ab的最短距离"""
    if np.all(a == b): return np.linalg.norm(p - a)
    ap, ab = p - a, b - a
    t = np.dot(ap, ab) / np.dot(ab, ab)
    if t < 0.0: return np.linalg.norm(p - a)
    if t > 1.0: return np.linalg.norm(p - b)
    return np.linalg.norm(p - (a + t * ab))


drone_speed, drone_angle, drone_flying_time, bomb_burst_time = 120, np.pi, 1.5, 3.6
# 无人机方向、角度、飞行时间、起爆时间
    
fake_target_location = np.array([0,0,0]) # 假目标位置
real_target_location = np.array([0,200,10]) # 真目标下平面圆心位置
drone_start_location  = np.array([17800,0,1800]) # 无人机起始位置

drone_direction = np.array([np.cos(drone_angle),np.sin(drone_angle),0]) # 无人机方向向量

bomb_start_location = drone_start_location + drone_direction * drone_speed * drone_flying_time # 烟雾弹落下位置

bomb_burst_location = bomb_start_location + drone_direction * drone_speed * bomb_burst_time + 0.5 * np.array([0,0,G]) * bomb_burst_time**2 # 烟雾弹爆炸位置

M1_start_location = np.array([20000,0,2000]) # 导弹起始位置
M1_speed = 300 # 导弹速度
M1_direction = (fake_target_location - M1_start_location)/np.linalg.norm(fake_target_location - M1_start_location) # 导弹方向向量

M1_count_location = M1_start_location + M1_direction * (drone_flying_time + bomb_burst_time) * M1_speed # 烟雾弹爆炸时导弹位置


cover = False
t = 0
delta_t = 0.005 # 时间步长
bomb_speed = 3 # 烟雾下落速度
bomb_direction = np.array([0,0,-1]) # 烟雾下落方向向量
bomb_location = bomb_burst_location.copy()
M1_location = M1_count_location.copy()
cover_time = 0
while(1):
    
    bomb_location += delta_t * bomb_speed * bomb_direction
    M1_location += delta_t * M1_speed * M1_direction

    if M1_location[2] < bomb_location[2]- 10 or t >= 20:
        # 导弹位置低于烟雾位置或时间超过爆炸20s后结束计算
        if cover:
            end_time = t
            cover_time += end_time - start_time
        break
    
    distance = distance_point_to_line_segment(bomb_location, M1_location, real_target_location) # 导弹与真目标连线到烟雾中心距离
    if not cover and distance <= 10:
        cover = True
        start_time = t
    if cover and distance > 10:
        cover = False
        end_time = t
        cover_time += end_time - start_time
    t += delta_t

print(f'遮蔽时间:{cover_time} s')


遮蔽时间:1.4599999999999689 s


In [2]:
import numpy as np
G = -9.8

def distance_point_to_line_segment(p, a, b):
    """计算点p到线段ab的最短距离"""
    if np.all(a == b): return np.linalg.norm(p - a)
    ap, ab = p - a, b - a
    t = np.dot(ap, ab) / np.dot(ab, ab)
    if t < 0.0: return np.linalg.norm(p - a)
    if t > 1.0: return np.linalg.norm(p - b)
    return np.linalg.norm(p - (a + t * ab))

def get_cover_time(particle:np.ndarray, missile_flying_time, output=False):
    """
    计算给定无人机方向、角度、飞行时间和起爆时间时的目标被遮蔽的总时间
    :param particle: 含有参数的粒子，[drone_speed, drone_angle, drone_flying_time, bomb_burst_time]
    :param missile_flying_time: 导弹飞行总时间
    :param output:控制是否输出
    :return: 目标被遮蔽总时间
    """
    drone_speed, drone_angle, drone_flying_time, bomb_burst_time = particle.tolist()
    # 无人机方向、角度、飞行时间、起爆时间
    
    # 若无人机飞行时间+起爆时间>=导弹飞行时间，遮蔽时间为0
    if drone_flying_time + bomb_burst_time >= missile_flying_time:
        return 0
     
    fake_target_location = np.array([0,0,0]) # 假目标位置
    real_target_location_points = []
    for z0 in [0,10]:
        for i in range(0, 100):
            real_target_location_points.append(np.array([np.cos(i/50*np.pi),200+np.sin(i/50*np.pi),z0]))
    drone_start_location  = np.array([17800,0,1800]) # 无人机起始位置

    drone_direction = np.array([np.cos(drone_angle),np.sin(drone_angle),0]) # 无人机方向向量

    bomb_start_location = drone_start_location + drone_direction * drone_speed * drone_flying_time # 烟雾弹落下位置
    
    bomb_burst_location_x = bomb_start_location[0] + drone_direction[0] * drone_speed * bomb_burst_time
    bomb_burst_location_y = bomb_start_location[1] + drone_direction[1] * drone_speed * bomb_burst_time
    bomb_burst_location_z = bomb_start_location[2] + G * bomb_burst_time**2 / 2
    bomb_burst_location = np.array([bomb_burst_location_x, bomb_burst_location_y, bomb_burst_location_z]) # 烟雾弹爆炸位置
    
    M1_start_location = np.array([20000,0,2000]) # 导弹起始位置
    M1_speed = 300 # 导弹速度
    M1_direction = (fake_target_location - M1_start_location)/np.linalg.norm(fake_target_location - M1_start_location) # 导弹方向向量

    M1_count_location = M1_start_location + M1_direction * (drone_flying_time + bomb_burst_time) * M1_speed # 烟雾弹爆炸时导弹位置

    if output:
        print(f'烟雾弹抛下位置:{bomb_start_location}')
        print(f'烟雾弹爆炸位置:{bomb_burst_location}')
        print(f'导弹方向:{M1_direction}')
        print(f'烟雾弹爆炸时导弹位置:{M1_count_location}')

    cover = False
    t = 0
    delta_t = 0.005 # 时间步长
    bomb_speed = 3 # 烟雾下落速度
    bomb_direction = np.array([0,0,-1]) # 烟雾下落方向向量
    bomb_location = bomb_burst_location.copy()
    M1_location = M1_count_location.copy()
    cover_time = 0
    while(1):
        
        bomb_location += delta_t * bomb_speed * bomb_direction
        M1_location += delta_t * M1_speed * M1_direction

        if M1_location[2] < bomb_location[2]- 10 or t >= 20:
            # 导弹位置低于烟雾位置或时间超过爆炸20s后结束计算
            if cover:
                end_time = t
                cover_time += end_time - start_time
            break
        
        all_in = True
        for point in real_target_location_points:
            distance = distance_point_to_line_segment(bomb_location, M1_location, point) # 导弹与真目标连线到烟雾中心距离
            if distance > 10:
                all_in = False
                
        if not cover and all_in:
            cover = True
            start_time = t
        if cover and not all_in:
            cover = False
            end_time = t
            cover_time += end_time - start_time
        t += delta_t
    return cover_time

print(f'遮蔽时间:{get_cover_time(np.array([120, np.pi, 1.5, 3.6]),missile_flying_time=67,output=True):.3f} s')

烟雾弹抛下位置:[1.76200000e+04 2.20436424e-14 1.80000000e+03]
烟雾弹爆炸位置:[1.71880000e+04 7.49483841e-14 1.73649600e+03]
导弹方向:[-0.99503719  0.         -0.09950372]
烟雾弹爆炸时导弹位置:[18477.59309898     0.          1847.7593099 ]
遮蔽时间:1.405 s


In [15]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def cone_surface_equation(vertex, base_center, radius):
    """生成圆锥面方程的符号表达式"""
    x, y, z = sp.symbols('x y z')
    x0, y0, z0 = vertex.tolist()
    x1, y1, z1 = base_center.tolist()
    
    # 计算向量VC和VP
    VC_x = x1 - x0
    VC_y = y1 - y0
    VC_z = z1 - z0
    VP_x = x - x0
    VP_y = y - y0
    VP_z = z - z0
    
    # 计算关键量
    VC_sq = VC_x**2 + VC_y**2 + VC_z**2  # |VC|^2
    dot_product = VC_x*VP_x + VC_y*VP_y + VC_z*VP_z  # VC · VP
    VP_sq = VP_x**2 + VP_y**2 + VP_z**2  # |VP|^2
    
    # 构造圆锥面方程
    k1 = VC_sq**2  # |VC|^4
    k2 = VC_sq + radius**2  # |VC|^2 + r^2
    equation = (dot_product**2) * k2 - VP_sq * k1
    
    return sp.simplify(equation)

def find_intersection_curve(cone_eq, plane_eq, output=False):
    """求解圆锥面与平面的交线"""
    x, y, z = sp.symbols('x y z')

    # 从平面方程解出一个变量
    if plane_eq.has(z):
        z_expr = sp.solve(plane_eq, z)[0]
        # 代入圆锥面方程
        projection_eq = cone_eq.subs(z, z_expr)
        projection_eq = sp.simplify(projection_eq)
        if output:
            print("\n交线在xy平面的投影方程:")
            print(projection_eq, "= 0")
        
        # 创建参数方程
        t = sp.symbols('t')
        
        # 尝试解出y关于x的关系
        try:
            y_solutions = sp.solve(projection_eq, y)
            if y_solutions:
                # 使用x作为参数
                parametric_x = t
                parametric_y = y_solutions[0].subs(x, t)
                parametric_z = z_expr.subs({x: t, y: parametric_y})
                
                if output:
                    print("\n参数方程 (x作为参数):")
                    print(f"x = {t}")
                    print(f"y = {sp.simplify(parametric_y)}")
                    print(f"z = {sp.simplify(parametric_z)}")
                
                return projection_eq, (parametric_x, parametric_y, parametric_z)
        except:
            pass
        
        # 如果解y失败，尝试解x
        try:
            x_solutions = sp.solve(projection_eq, x)
            if x_solutions:
                # 使用y作为参数
                parametric_x = x_solutions[0].subs(y, t)
                parametric_y = t
                parametric_z = z_expr.subs({x: parametric_x, y: t})
                
                print("\n参数方程 (y作为参数):")
                print(f"x = {sp.simplify(parametric_x)}")
                print(f"y = {t}")
                print(f"z = {sp.simplify(parametric_z)}")
                
                return projection_eq, (parametric_x, parametric_y, parametric_z)
        except:
            pass
        
        # 两种方法都失败时，返回投影方程
        return projection_eq, None
    
    return None, None

In [16]:
# 定义符号变量
x, y, z = sp.symbols('x y z')

cone_eq = cone_surface_equation(M1_count_location, bomb_burst_location, 10.0)

print("圆锥面方程:")
print(cone_eq, "= 0")

# 定义平面方程
plane_eq = z 

print("\n平面方程:")
print(plane_eq, "= 0")

# 求解交线
projection_eq, param_eq = find_intersection_curve(cone_eq, plane_eq)

圆锥面方程:
-20574719653.2114*x**2 + 480824789133.381*x*z - 128105884795696.0*x - 2807065299762.07*y**2 - 2786323037120.36*z**2 + 1.41242385893525e+15*z - 1.21365461015204e+17 = 0

平面方程:
z = 0


In [17]:
from scipy.optimize import minimize

real_target_location_up = np.array([0,200,10])

center = sp.solve([projection_eq.diff(x),projection_eq.diff(y)],(x,y))
print(center)

def min_distance_to_ellipse(x0, y0, eq, num_initial_points=8, tol=1e-6):
    """
    计算点(x0, y0)到椭圆的最小距离（数值方法）
    
    参数:
    x0, y0 -- 目标点坐标
    eq -- 椭圆方程（Sympy表达式），形式为 eq = 0
    num_initial_points -- 初始点数量（默认8）
    tol -- 优化容差（默认1e-6）
    
    返回:
    最短距离
    """
    # 将符号表达式转换为数值函数
    x, y = sp.symbols('x y')
    ellipse_func = sp.lambdify((x, y), eq, 'numpy')
    
    # 定义目标函数（距离的平方）
    def objective(params):
        """最小化点(x,y)到(x0,y0)的距离平方"""
        return (params[0] - x0)**2 + (params[1] - y0)**2
    
    # 定义约束条件（点必须在椭圆上）
    def constraint(params):
        """约束条件：点必须在椭圆上（eq=0）"""
        return ellipse_func(params[0], params[1])
    
    # 创建约束对象
    cons = {'type': 'eq', 'fun': constraint}
    
    # 生成初始点（均匀分布在椭圆上）
    initial_points = []
    for i in range(num_initial_points):
        angle = 2 * np.pi * i / num_initial_points
        # 使用角度作为初始猜测（可能不在椭圆上）
        initial_points.append([np.cos(angle), np.sin(angle)])
    
    # 添加目标点方向作为额外初始点
    if x0 != 0 or y0 != 0:
        direction = np.array([x0, y0])
        direction = direction / np.linalg.norm(direction)
        initial_points.append(direction.tolist())
    
    # 对每个初始点进行优化
    min_distance = float('inf')
    for point in initial_points:
        # 使用SLSQP算法进行约束优化
        result = minimize(objective, point, 
                          constraints=cons,
                          method='SLSQP',
                          tol=tol)
        
        # 只考虑成功的优化结果
        if result.success:
            # 计算实际距离（平方根）
            distance = np.sqrt(result.fun)
            if distance < min_distance:
                min_distance = distance
                best_point = result.x
    
    if min_distance == float('inf'):
        raise RuntimeError("优化失败，未找到有效解")
    
    return min_distance

print(min_distance_to_ellipse(real_target_location_up[0],real_target_location_up[1],projection_eq))

{x: -3113.18664251399, y: 0.0}
1182.4114730037434


In [19]:
def target_in_shadow(missile_location:np.ndarray, bomb_location:np.ndarray, target_location:tuple):
    """判断真目标圆柱是否在投影范围内"""

    reason = '无'

    if np.linalg.norm(missile_location - bomb_location) <= 10:
        # 导弹在烟雾范围内
        return True, reason
    else:
        # 计算投影圆锥面方程
        x, y, z = sp.symbols('x y z')
        cone_eq = cone_surface_equation(vertex=missile_location, base_center=bomb_location, radius=10)

        x0 = target_location[0]
        y0 = target_location[1]

        if missile_location[2] > bomb_location[2] + 10:
            # 导弹位置在烟雾上方，此时圆锥面与水平面交线为椭圆
            for z0 in [0, 10]:
                # 判断圆柱上下底面是否在投影内
                plane_eq = z - z0 # 水平面
                ellipse_eq, _ = find_intersection_curve(cone_eq, plane_eq) # 求解交线
                ellipse_func = sp.lambdify((x, y), ellipse_eq, 'numpy')
                center = sp.solve([ellipse_eq.diff(x),ellipse_eq.diff(y)],(x,y))
                f_center = ellipse_func(center[x], center[y])
                f_x0y0 = ellipse_func(x0, y0)
                
                if f_center * f_x0y0 < 0:
                    # 圆心在椭圆外部
                    reason = f'圆心在椭圆外部,{z0}'
                    return False, reason
                else:
                    # 圆心在椭圆内部，计算圆心到椭圆的最小距离
                    min_d = min_distance_to_ellipse(x0, y0, ellipse_eq)
                    if min_d < 10:
                        reason = f'圆心在椭圆内部,{z0}'
                        return False, reason
        else:
            # 导弹位置和烟雾重合，此时圆锥面与水平面交线为双曲线，需要考虑双曲线的一支
            # 判断圆柱下底面是否在投影内
            plane_eq = z # 水平面
            ellipse_eq, _ = find_intersection_curve(cone_eq, plane_eq) # 求解交线
            ellipse_func = sp.lambdify((x, y), ellipse_eq, 'numpy')
            center = sp.solve([ellipse_eq.diff(x),ellipse_eq.diff(y)],(x,y))
            f_center = ellipse_func(center[x], center[y])
            f_x0y0 = ellipse_func(x0, y0)

            if f_center * f_x0y0 > 0:
                # 圆心在双曲线内侧
                reason = '圆心在双曲线内侧'
                return False, reason
            else:
                # 圆心在双曲线外侧，计算圆心到双曲线的最小距离
                min_d = min_distance_to_ellipse(x0, y0, ellipse_eq)
                if min_d < 10:
                    reason = '圆心在双曲线外侧'
                    return False, reason
                
        return True, reason

cover = False
t = 0
delta_t = 0.005
bomb_speed = 3
bomb_direction = np.array([0,0,-1])
bomb_location = bomb_burst_location.copy()
M1_location = M1_count_location.copy()
cover_time = 0
while(1):
    bomb_location += delta_t * bomb_speed * bomb_direction
    M1_location += delta_t * M1_speed * M1_direction

    if M1_location[2] < bomb_location[2]- 10 or t >= 20:
        # 导弹位置低于烟雾位置或时间超过爆炸20s后结束计算
        print(res[1])
        print(M1_location)
        print(bomb_location)
        print('over')
        if cover:
            end_time = t
            cover_time += end_time - start_time
        break

    res = target_in_shadow(M1_location, bomb_location, (0,200))
    if not cover and res[0]:
        cover = True
        start_time = t
        print(start_time)
    if cover and not res[0]:
        cover = False
        end_time = t
        cover_time += end_time - start_time
        print(res[1])
        print(M1_location)
        print(bomb_location)
    t += delta_t

print(f'遮蔽时长:{cover_time} s')


2.964999999999959
圆心在双曲线内侧
[17129.81522484     0.          1712.98152248]
[17188.        0.     1722.951]
圆心在双曲线内侧
[17128.32266905     0.          1712.83226691]
[17188.        0.     1722.936]
over
遮蔽时长:1.544999999999967 s
